## Imports

In [13]:
from typing import List, Any
import pydantic_ai as pda
import os
import pickle
import json
import secrets
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import numpy as np
import openai
from sentence_transformers import SentenceTransformer

load_dotenv()

True

In [2]:
def load_object(file_path):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File {file_path} does not exist")
    
    if not file_path.endswith('.pkl'):
        raise ValueError(f"File {file_path} is not a pickle file")
    
    with open(file_path, 'rb') as f:
        return pickle.load(f)

## Agent

In [3]:
section_index =  load_object("data/section_chunks_lexical_index.pkl")

In [4]:
def lexical_search(query: str) -> List[Any]:
    return section_index.search(query, num_results=5)

In [69]:
vector_index = load_object('data/section_chunks_vec_index.pkl')
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

In [6]:
def vector_search(query):
    query_emb = embedding_model.encode(query)
    return vector_index.search(query_emb, num_results=5)

In [7]:
openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [25]:
system_prompt = """
    You are a helpful assistant for a course.
    Use the search tool to find relevant information from the course materials
    before answering questions.
    If you can find specific information through search, state that you have gathered some relevant
    information from the search, explicitly include references under "References:" at the top of your answer
    to that information by citing something like the filename, the URL to the github repo or the section name
    for the chunk(s) you used info from and use it to provide accurate answers.
    Always include the references at the top of your answer in the below format.
    FORMAT: 
    References:
    1. [REPO NAME]-[SECTION NAME] -> [FULL GITHUB URL]
    2. [REPO NAME]-[SECTION NAME] -> [FULL GITHUB URL]
    ...
    
    If the search doesn't return relevant results, let the user know that and provide
    general guidance based on your existing knowledge.
"""

In [9]:
user_prompt = """
    What are the best practices for configuring HPA (Horizontal Pod Autoscaler) in production environments, 
    and what metrics should we consider for optimal scaling?
"""

In [26]:
agent = pda.Agent(
    name="Tech Assistant",
    instructions=system_prompt,
    tools=[lexical_search],
    model="gpt-4o-mini"
)

In [11]:
result = await agent.run(user_prompt=user_prompt)

## Logging

In [12]:
def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())
    dict_messages = pda.ModelMessagesTypeAdapter.dump_python(messages)
    return {
        'agent_name': agent.name,
        'system_prompt': agent._instructions,
        'provider': agent.model.system,
        'model': agent.model.model_name,
        'tools': tools,
        'messages': dict_messages,
        'source': source
    }

In [14]:
LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

In [15]:
def serialize_log(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

In [16]:
def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)
    
    ts = entry['messages'][-1]['timestamp']
    ts_str = ts.strftime("%Y-%m-%d %H:%M:%S")
    
    random_hex = secrets.token_hex(3)
    
    filename = f"{agent.name}_{ts_str}_{random_hex}.json"
    filepath = LOG_DIR / filename
    
    with filepath.open("w", encoding="utf-8") as f:
        json.dump(entry, f, indent=2, default=serialize_log)
    
    return filepath
    
    

In [27]:
question = "What is Reinforcement Learning?"
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

# Ideal Answer:
# Reinforcement learning, RL, is seen as one of the basic machine learning paradigms, 
# next to supervised learning and unsupervised learning. RL is all about decisions: 
# delivering the right decisions or at least learning from them.
# Imagine you have a simulated environment such as the stock market. What happens 
# if you impose a given regulation? Does it have a positive or negative effect? 
# If something negative happens, you need to take this negative reinforcement, 
# learn from it, and change course. If it's a positive outcome, you need to build 
# on that positive reinforcement.

References:
1. [ml-for-beginners-main-8-reinforcement](https://github.com/microsoft/ML-For-Beginners/blob/main/8-reinforcement/readme.md)
2. [ml-for-beginners-main-8-reinforcement-1-qlearning](https://github.com/microsoft/ML-For-Beginners/blob/main/translations/en/8-reinforcement/1-qlearning/readme.md)

Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by taking actions in an environment to maximize some notion of cumulative reward. The core concepts of reinforcement learning include:

- **States**: The current situation of the agent within the environment.
- **Actions**: Choices made by the agent that affect its state.
- **Rewards**: Feedback from the environment following an action, guiding the agent in its learning process.
- **Policies**: Strategy that defines the agent's way of behaving at a given time.

An important mechanism in reinforcement learning is the **reward function**, which provides feedback to the agent regarding the suc

PosixPath('logs/Tech Assistant_2025-10-03 14:47:09_ec55d9.json')

## Eval using an LLM as judge

In [28]:
from pydantic import BaseModel

In [ ]:
class EvalsCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool
    
class EvalsChecklist(BaseModel):
    checks: List[EvalsCheck]
    summary: str

In [40]:
evals_system_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>)
to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.
For each item, check if the condition is met.
Checklist:
- instructions_follow: The agent followed the user's instructions (in
<INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when
required
- completeness: The response is complete and covers all key aspects of the
request
- tool_call_search: Is the search tool invoked?
Output true/false for each check and provide a short explanation for your
judgment

Then include a summary (true/false) of whether this eval case has 
passed/failed considering the entire checklist.
""".strip()

In [41]:
eval_agent = pda.Agent(
    name="eval-agent",
    instructions=evals_system_prompt,
    model="gpt-3.5-turbo",
    output_type=EvalsChecklist
)

In [42]:
eval_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [43]:
def load_log_file(log_file):
    with open(log_file, "r") as f:
        log_data = json.load(f)
        log_data['log_file'] = log_file
    return log_data

In [44]:
evals_summary = []
for log_file in os.listdir(LOG_DIR):
    if not log_file.startswith("Tech Assistant") or not log_file.endswith(".json"):
        continue
    log_data = load_log_file(os.path.join(LOG_DIR, log_file))
    instructions = log_data["system_prompt"]
    question = log_data["messages"][0]["parts"][0]["content"]
    answer = log_data["messages"][-1]["parts"][0]["content"]
    log = json.dumps(log_data["messages"])
    
    eval_prompt = eval_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )
    
    eval_response = await eval_agent.run(eval_prompt, output_type=EvalsChecklist)
    eval = {
        "question": question,
        "answer": answer,
        "eval": eval_response.output.summary
    }
    
    evals_summary.append(eval)

In [45]:
evals_summary[0]

{'question': 'How does autoscaling work in kubernetes?',
 'answer': "It seems that I couldn't find specific information regarding autoscaling in Kubernetes within the course materials. However, I can provide a general overview of how autoscaling works in Kubernetes.\n\n### Overview of Autoscaling in Kubernetes\n\nKubernetes supports autoscaling through two primary mechanisms:\n\n1. **Horizontal Pod Autoscaler (HPA)**:\n   - HPA automatically adjusts the number of pod replicas in a deployment based on observed CPU utilization or other select metrics.\n   - It periodically queries the metrics server for resource usage and scales up or down the number of pods to ensure that resource requests are met.\n   - You can define thresholds that trigger scaling actions (e.g., scaling up if CPU usage exceeds 80%).\n\n2. **Vertical Pod Autoscaler (VPA)**:\n   - VPA automatically adjusts the resource requests and limits for containers in a pod based on usage.\n   - Instead of changing the number of r

In [47]:
correct = len([e for e in evals_summary if e['eval'].lower() == 'true'])
total = len(evals_summary)
accuracy = correct / total * 100

print(f"""
EVALS SUMMARY:\n\n
Total Questions: {total}
Correct Answers: {correct}
Incorrect Answers: {total - correct}
Accuracy: {accuracy:.2f}%
""")


EVALS SUMMARY:


Total Questions: 7
Correct Answers: 7
Incorrect Answers: 0
Accuracy: 100.00%

